<a href="https://colab.research.google.com/github/DaeYejun2/ML_Notes/blob/main/ml_assignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1: Linear Regression

**Due:** 4/26 (Sun.) 23:59

## Objectives
- Explore and visualize the dataset to understand feature-target relationships.
- Implement Simple and Multiple Linear Regression from scratch.
- Understand how Gradient Descent works.
- Implement feature normalization and understand its effect on convergence.
- Implement L2 regularization and understand how it controls model complexity.

⚠️ Only fill in the `# YOUR CODE HERE` blocks. Do not modify any other cells.

---

## Instructions

> 💡 New to Google Colab? Check out the [official introduction](https://colab.research.google.com/notebooks/basic_features_overview.ipynb).

⚠️ Before beginning, click **File → Save a copy in Drive** to fork this notebook to your Google Drive.

Submit both files compressed as a single ZIP to eLMS:
- Notebook: `.ipynb` file
  > 💡 To download the notebook: **File → Download → Download .ipynb**

- Report: `student_id_name.pdf` (e.g., `2024000001_홍길동.pdf`)
ZIP filename: `student_id_name.zip` (e.g., `2024000001_홍길동.zip`)

⚠️ Submissions not following the filename format will not be accepted.

---

## Report Guidelines

**Length:** Within 7 pages (Microsoft Word, converted to PDF)

**Required Sections:**
- Implementation (Part 3 & Part 4-1): explain each function + key code snippet
- Feature Normalization (Part 4-2): explain `normalize_zscore` + cost curve comparison plot
- Regularization (Part 4-4): explain `compute_cost_regularized` and `compute_gradients_regularized` + plots
- Screenshots: Sanity Check passed, all result plots
- Analysis Questions (Part 5, Q1–Q6): label each answer with its question number

**Grading (100pts):**
- Code: 50pts — Sanity Check screenshots (Parts 3-1, 4-2a, 4-4a) + all result plots
- Report: 50pts

---



## Part 1. Data Preparation

Before building the model, let's explore the data.
In this assignment, we use California housing dataset (see: https://inria.github.io/scikit-learn-mooc/python_scripts/datasets_california_housing.html)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

In [ ]:
housing = fetch_california_housing()
X_raw, y = housing.data, housing.target
feature_names = housing.feature_names

print(f"Data shape: {X_raw.shape}")
print(f"Features: {feature_names}")
print(f"Target (median house price, $100K) — first 5 samples: {y[:5]}")

### Part 1-1. NumPy Basics: Vectors and Shapes

⚠️ Read this subsection very carefully and run the code **before** proceeding to the next steps.

Before we dive into the model, let's practice how to handle arrays and shapes using NumPy. This is **crucial** for correctly implementing the cost and gradient functions.

> **Key Concept:**
> - `a.shape = (3,)` is a 1D Rank-1 array.
> - `b.shape = (3, 1)` is a 2D column vector.
> - Subtracting them directly can lead to unexpected results due to **broadcasting**.

In [ ]:
# 1. Creating arrays and checking shapes
a = np.array([1, 2, 3])          # 1D array (vector), shape: (3,)
b = np.array([[1], [2], [3]])    # 2D array (column vector), shape: (3, 1)

print(f"Shape of a: {a.shape}")
print(f"Shape of b: {b.shape}")

# 2. Accessing elements
# To get the scalar value '1' from 'a':
print(f"a[0]: {a[0]}")

# To get the scalar value '1' from 'b' (which is 2D):
print(f"b[0, 0]: {b[0, 0]}")
# OR using single index (returns a 1D array of size 1):
print(f"b[0]: {b[0]}, type: {type(b[0])}")

# 3. Arithmetic between different shapes

# Subtracting 1D array 'a' from a scalar works fine.
# But subtracting a 1D array from a 2D array requires caution (Broadcasting).
# Example: (3, 1) - (3,) might not result in what you expect if not handled correctly.

target = np.array([0.5, 1.5, 2.5])

# Method A: Use indexing [i][0] to get scalar (safe for loops)
error_loop = b[0][0] - target[0]

# Method B: Use .flatten() to make them both 1D
error_flat = b.flatten() - target

print(f"Loop error (scalar): {error_loop}")
print(f"Vector error (1D array): {error_flat}")

## Part 2. Exploratory Data Analysis (EDA)

Let's visualize the relationship between each feature and the target (housing price), and think about which features might be useful for prediction.

In [ ]:
# Scatter plot: each feature vs. target
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, name in enumerate(feature_names):
    axes[i].scatter(X_raw[:, i], y, alpha=0.1, s=5, color='steelblue')
    axes[i].set_xlabel(name, fontsize=10)
    axes[i].set_ylabel('Price ($100K)', fontsize=10)
    axes[i].set_title(f'{name} vs Price', fontsize=11)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Feature vs Housing Price (California Housing Dataset)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Pearson correlation of each feature with the target
correlations = [np.corrcoef(X_raw[:, i], y)[0, 1] for i in range(X_raw.shape[1])]

colors = ['tomato' if c < 0 else 'steelblue' for c in correlations]
plt.figure(figsize=(9, 4))
bars = plt.bar(feature_names, correlations, color=colors, edgecolor='white')
plt.axhline(0, color='black', linewidth=0.8)
plt.ylabel('Correlation with Price')
plt.title('Feature Correlation with Housing Price')
plt.xticks(rotation=20)
plt.grid(True, axis='y', alpha=0.3)
for bar, val in zip(bars, correlations):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.01 * np.sign(val),
             f'{val:.2f}', ha='center', va='bottom' if val > 0 else 'top', fontsize=9)
plt.tight_layout()
plt.show()

print("\nFeature correlation ranking (by absolute value):")
sorted_idx = np.argsort(np.abs(correlations))[::-1]
for rank, idx in enumerate(sorted_idx):
    print(f"  {rank+1}. {feature_names[idx]:12s}: {correlations[idx]:.4f}")

## Part 3. Simple Linear Regression (Single Feature)

In this section, we use only **'MedInc'** (Median Income) to predict housing prices.

### 3-1. Mathematical Background

Recall that we have learned linear regression model as follows:

$$f_{w,b}(x) = wx + b$$

And our cost function MSE (Mean Squared Error) is:

$$J(w, b) = \frac{1}{2m} \sum_{i=1}^{m} (f_{w,b}(x^{(i)}) - y^{(i)})^2$$

Note that we use $f_{w,b}$ and $ŷ$ interchangebly.

To this end, we use the gradient descent algorithm:

$$\text{Repeat until convergence:$$
$$} \{$$
$$\begin{aligned} w &= w - \alpha \cdot \frac{\partial}{\partial w}J(w,b) \\ b &= b - \alpha \cdot \frac{\partial}{\partial b}J(w,b) \end{aligned}$$
$$\} \text{ Update w and b simultaneously}$$

, where
$$\frac{\partial}{\partial w}J(w,b)=(f_{w,b}(x^{(i)}-y^{(i)})x^{(i)}$$
$$\frac{\partial}{\partial b}J(w,b)=(f_{w,b}(x^{(i)}-y^{(i)})$$

Now, your goal is to implement a linear regression model and gradient descents.

⚠️ IMPORTANT: Use for-loops to implement below code to understand the mechanics.

In [ ]:
### Step 1: Data Preparation
# Use only MedInc (index 0)
X_simple = X_raw[:, 0:1]

X_train_s, X_test_s, y_train, y_test = train_test_split(X_simple, y, test_size=0.2, random_state=42)

print(f"Train shape: {X_train_s.shape}, Test shape: {X_test_s.shape}")

In [ ]:
### Step 2: Implementation (Loop-based)

def predict(x, w, b):
    """
    Compute predictions for linear regression.
    Arguments:
        x: (m,) 1D array of input vectors
        w: scalar, weight
        b: scalar, bias
    Returns:
        y_hat: (m,) 1D array of predicted values
    """
    x = np.ravel(x)
    m = x.shape[0]
    y_hat = np.zeros(m)

    # YOUR CODE HERE
    # Iterate through m samples and calculate w*x + b

    return y_hat

In [ ]:
def compute_cost(y_hat, y):
    """
    Compute mean squared error cost.
    Arguments:
        y_hat: (m,) 1D array of predicted values
        y:     (m,) 1D array of ground truth values
    Returns:
        cost: scalar
    """
    m = y.shape[0]
    cost = 0

    # YOUR CODE HERE
    # Compute the average of squared errors using a loop

    return cost

In [ ]:
def compute_gradients(x, y_hat, y):
    """
    Compute gradients of the cost w.r.t. w and b using loops.
    x is (m,), y_hat and y are (m,)
    Returns:
        dj_dw: scalar gradient w.r.t. weights
        dj_db: scalar gradient w.r.t. bias
    """
    m = x.shape[0]
    dj_dw = 0.0
    dj_db = 0.0

    # YOUR CODE HERE
    # Iterate through m samples to accumulate gradients

    return dj_dw, dj_db

In [ ]:
def gradient_descent(x, y, learning_rate=0.01, epochs=1000):
    """
    Run gradient descent to learn w and b.
    Returns:
        w, b, cost_history
    """
    w = 0.0
    b = 0.0
    cost_history = []

    print(f"Starting Gradient Descent with learning_rate={learning_rate}, epochs={epochs}...")

    for epoch in range(epochs):
        # Step 1: Compute predictions
        # YOUR CODE HERE

        # Step 2: Compute cost
        # YOUR CODE HERE
        cost_history.append(cost)

        # Step 3: Compute gradients
        # YOUR CODE HERE

        # Step 4: Update parameters simultaneously
        # YOUR CODE HERE

        w = float(np.squeeze(w))
        b = float(np.squeeze(b))

        # Print progress every 100 epochs
        if epoch % 100 == 0:
            print(f"Epoch {epoch:4d}: Cost {cost:.6f}")

    print(f"Training finished. Final Cost: {cost_history[-1]:.6f}")
    return w, b, cost_history

### Part 3-1. Sanity Check (do not modify)

In [ ]:
# Sanity check — do not modify this cell
X_dummy = np.array([1.0, 2.0, 3.0])
w_dummy = 2.0
b_dummy = 1.0

y_hat = predict(X_dummy, w_dummy, b_dummy)
assert y_hat.shape == (3,) , f"predict: wrong shape {y_hat.shape}"
assert np.allclose(y_hat.flatten(), [3.0, 5.0, 7.0]), f"predict: wrong values {y_hat.flatten()}"

y_dummy = np.array([3.0, 5.0, 8.0])
cost = compute_cost(y_hat, y_dummy)
assert np.isscalar(cost) or cost.ndim == 0, "cost must be a scalar"
assert np.isclose(cost, 1/6), f"compute_cost: wrong value {cost} (expected {1/6:.6f})"

dj_dw, dj_db = compute_gradients(X_dummy, y_hat, y_dummy)
assert np.isscalar(dj_dw), f"compute_gradients: dj_dw must be a scalar"
assert np.isclose(dj_db, -1/3), f"compute_gradients: wrong db {dj_db} (expected {-1/3:.6f})"

print("✅ All checks passed!")

In [ ]:
w_s, b_s, cost_history_s = gradient_descent(X_train_s, y_train, learning_rate=0.01, epochs=1000)
print(f"Simple LR — w: {w_s}, b: {b_s:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Cost curve
axes[0].plot(cost_history_s, color='steelblue')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cost J(w,b)')
axes[0].set_title('Training Cost Curve (Simple LR)')
axes[0].grid(True, alpha=0.3)

# Regression fit
y_pred_s = predict(X_test_s, w_s, b_s)
sort_idx = np.argsort(X_test_s[:, 0])
axes[1].scatter(X_test_s, y_test, alpha=0.3, s=8, label='Actual', color='steelblue')
axes[1].plot(X_test_s[sort_idx], y_pred_s[sort_idx], color='tomato', linewidth=2, label='Predicted')
axes[1].set_xlabel('MedInc (normalized)')
axes[1].set_ylabel('Price ($100K)')
axes[1].set_title('Simple Linear Regression Fit')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

mse_s = np.mean((y_pred_s.flatten() - y_test) ** 2)
print(f"Simple LR — Test MSE: {mse_s:.4f}")
plt.tight_layout()
plt.show()

---
## Part 4. Multiple Linear Regression (all features)

Now we use all 8 features for housing price prediction.

Recall that we have learned multiple linear regression model that takes $n$ features as follows:

$$f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w⃗} \cdot \mathbf{x} + b = w_1x_1 + w_2x_2 + \cdots + w_nx_n + b$$

And then our cost function is:

$$J(\mathbf{w⃗}, b) = \frac{1}{2m} \sum_{i=1}^{m} (f_{\mathbf{w⃗},b}(\mathbf{x}^{(i)}) - y^{(i)})^2$$

Finally, the gradient descent is changed as:

$$\text{Repeat until convergence:} \{$$

$$w_j = w_j - \alpha \cdot \frac{\partial}{\partial w_j}J(\mathbf{w⃗}, b), \quad \frac{\partial}{\partial w_j}J(\mathbf{w⃗}, b) = \frac{1}{m}\sum_{i=1}^{m}(f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_j^{(i)}$$

$$b = b - \alpha \cdot \frac{\partial}{\partial b}J(\mathbf{w⃗}, b), \quad \frac{\partial}{\partial b}J(\mathbf{w⃗}, b) = \frac{1}{m}\sum_{i=1}^{m}(f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})$$
$$\} \text{Update w and b simultaneously}$$

---
## Part 4-1. Vectorization

In real-world ML, we avoid `for-loops` because they are slow on large datasets, especially when we handle multiple features.

Re-implement `compute_cost` and `compute_gradients` using **fully vectorized NumPy operations** (no loops).

Note that now the `X` is an input matrix `(m,n)` and `w` is an `(n,)` vector.

> 💡 Hint: Think about how to use `np.dot()`, `np.sum()`, and shape broadcasting instead of iterating over samples.

In [ ]:
def predict_vectorized(X, w, b):
    """
    Predict using multiple linear regression.
    X: (m, n) matrix, w: (n,) 1D array, b: scalar
    Returns 1D array (m,)
    """
    # YOUR CODE HERE
    # Use np.dot() and ensure the result is a 1D array

    return y_hat

def compute_cost_vectorized(y_hat, y):
    """
    Compute cost for multiple linear regresion.
    y_hat, y: (m,) 1D arrays
    Returns cost: scalar
    """

    m = y.shape[0]
    cost = 0
    # YOUR CODE HERE
    # Use np.sum() for efficiency

    return cost


def compute_gradients_vectorized(X, y_hat, y):
    """
    Compute gradients for multiple linear regression.
    X: (m, n), y_hat: (m,), y: (m,)
    Returns dj_dw: (n,) 1D array, dj_db: scalar
    """
    m = X.shape[0]
    n = X.shape[1]

    # YOUR CODE HERE
    # Use X.T to transpose matrix X in order to multiply with the error vector (y_hat - y)

    return dj_dw, dj_db

In [ ]:
# Use all features
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_raw, y, test_size=0.2, random_state=42)

scaler_m = StandardScaler()
X_train_m = scaler_m.fit_transform(X_train_m)
X_test_m  = scaler_m.transform(X_test_m)

print(f"Train: {X_train_m.shape}, Test: {X_test_m.shape}")
print(f"Features: {feature_names}")

# Verification of scaling (Should be Mean ~0 and Std ~1)
print(f"\nScaled Data Mean: {np.mean(X_train_m):.4f}")
print(f"Scaled Data Std:  {np.std(X_train_m):.4f}")

In [ ]:
def gradient_descent_vectorized(X, y, learning_rate=0.01, epochs=1000):
    """
    Gradient descent for multiple linear regression
    """
    n = X.shape[1]
    w = np.zeros(n)  # Initialize as 1D array (n,)
    b = 0.0
    cost_history = []

    print(f"Starting Gradient Descent with learning_rate={learning_rate}, epochs={epochs}...")

    for epoch in range(epochs):
        # Step 1: Compute predictions (m,)
        # YOUR CODE HERE

        # Step 2: Compute cost
        # YOUR CODE HERE
        cost_history.append(cost)

        # Step 3: Compute gradients
        # YOUR CODE HERE

        # Step 4: Update parameters
        # YOUR CODE HERE

        if epoch % 100 == 0:
            print(f"Epoch {epoch:4d}: Cost {cost:.6f}")

    return w, b, cost_history

In [ ]:
### Step 5: Run Multiple Linear Regression
w_m, b_m, cost_history_m = gradient_descent_vectorized(X_train_m, y_train_m, learning_rate=0.01, epochs=1000)

print("\nLearned weights per feature:")
for name, weight in zip(feature_names, w_m):
    print(f"  {name:12s}: {weight:+.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Cost curve comparison
axes[0].plot(cost_history_s, label='Simple LR (MedInc only)', color='steelblue')
axes[0].plot(cost_history_m, label='Multiple LR (all features)', color='tomato')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cost J(w,b)')
axes[0].set_title('Cost Curve: Simple vs Multiple LR')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Predicted vs Actual
y_pred_m = predict_vectorized(X_test_m, w_m, b_m)
lims = [min(y_test_m.min(), float(y_pred_m.min())), max(y_test_m.max(), float(y_pred_m.max()))]
axes[1].scatter(y_test_m, y_pred_m, alpha=0.3, s=8, color='tomato')
axes[1].plot(lims, lims, 'k--', linewidth=1, label='Perfect fit')
axes[1].set_xlabel('Actual Price ($100K)')
axes[1].set_ylabel('Predicted Price ($100K)')
axes[1].set_title('Multiple LR: Predicted vs Actual')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

mse_m = np.mean((y_pred_m.flatten() - y_test_m) ** 2)
print(f"Simple   LR — Test MSE: {mse_s:.4f}")
print(f"Multiple LR — Test MSE: {mse_m:.4f}")
plt.tight_layout()
plt.show()

---
## Part 4-2. Feature Normalization

In Part 4-1, we used `StandardScaler` to normalize features before training Multiple LR.  
But what does it actually do, and why does it matter?

### Why Normalization Matters

With 8 features that have very different scales (e.g., `Population` in thousands vs. `HouseAge` in years),  
the cost function forms a **stretched ellipse** in weight space. This forces gradient descent to zig-zag  
and converge slowly — or require a tiny learning rate.

Z-score normalization transforms each feature to have **mean ≈ 0** and **std ≈ 1**:

$$x_{j,\text{norm}} = \frac{x_j - \mu_j}{\sigma_j}$$

**Key rule:** Always compute μ and σ from the **training set only**, then apply to both train and test.

In [ ]:
def normalize_zscore(X_train, X_test):
    """
    Perform Z-score normalization (standardization) from scratch.
    Arguments:
        X_train: (m_train, n) 2D array of training features
        X_test:  (m_test, n)  2D array of test features
    Returns:
        X_train_norm: normalized training data  (m_train, n)
        X_test_norm:  normalized test data       (m_test, n)
        mu:    mean of each feature from training data  (n,)
        sigma: std  of each feature from training data  (n,)

    Hint:
        mu    = np.mean(X_train, axis=0)   # per-feature mean, shape (n,)
        sigma = np.std(X_train,  axis=0)   # per-feature std,  shape (n,)
        normalized = (X - mu) / sigma      # broadcasting handles (m, n) - (n,)
    """
    mu = np.zeros(X_train.shape[1])
    sigma = np.ones(X_train.shape[1])
    X_train_norm = X_train.copy()
    X_test_norm  = X_test.copy()

    # Step 1: Compute mu and sigma from X_train only (axis=0 → per feature)
    # YOUR CODE HERE

    # Step 2: Normalize both X_train and X_test using mu and sigma
    # YOUR CODE HERE

    return X_train_norm, X_test_norm, mu, sigma

### Part 4-2a. Sanity Check (do not modify)

In [ ]:
# Sanity check — do not modify this cell
np.random.seed(1)
X_dummy_train = np.random.randn(10, 3) * np.array([100, 5, 0.5])  # different scales
X_dummy_test  = np.random.randn(4,  3) * np.array([100, 5, 0.5])

X_tr_n, X_te_n, mu_d, sigma_d = normalize_zscore(X_dummy_train, X_dummy_test)

assert mu_d.shape == (3,),    f"mu should be shape (3,), got {mu_d.shape}"
assert sigma_d.shape == (3,), f"sigma should be shape (3,), got {sigma_d.shape}"
assert np.allclose(mu_d, np.mean(X_dummy_train, axis=0)), \
    f"mu wrong: got {mu_d}, expected {np.mean(X_dummy_train, axis=0)}"
assert np.allclose(sigma_d, np.std(X_dummy_train, axis=0)), \
    f"sigma wrong"
assert np.allclose(np.mean(X_tr_n, axis=0), 0, atol=1e-6), \
    f"Training mean should be ~0 per feature, got {np.mean(X_tr_n, axis=0)}"
assert np.allclose(np.std(X_tr_n, axis=0), 1, atol=1e-6), \
    f"Training std should be ~1 per feature, got {np.std(X_tr_n, axis=0)}"
# Test set uses TRAINING statistics
expected_te = (X_dummy_test - mu_d) / sigma_d
assert np.allclose(X_te_n, expected_te), "Test normalization wrong — use training mu/sigma"

print("✅ normalize_zscore: All checks passed!")

### Part 4-2b. Training Comparison: Without vs. With Normalization

In [ ]:
# Split raw (un-scaled) data
X_tr_raw, X_te_raw, y_tr_raw, y_te_raw = train_test_split(
    X_raw, y, test_size=0.2, random_state=42
)

# Apply your normalize_zscore
X_tr_norm2, X_te_norm2, mu_all, sigma_all = normalize_zscore(X_tr_raw, X_te_raw)

print("Feature statistics (training set):")
for name, m, s in zip(feature_names, mu_all, sigma_all):
    print(f"  {name:12s}: mean={m:8.3f}, std={s:7.3f}")

# ⚠️ Without normalization, lr=0.01 causes gradient explosion (cost → NaN).
# This is intentional — it demonstrates exactly why normalization is necessary.
# Notice that With Normalization converges cleanly with the same lr.
print("\n--- Without Normalization ---")
w_no,  b_no,  hist_no  = gradient_descent_vectorized(
    X_tr_raw,   y_tr_raw, learning_rate=0.01, epochs=1000)

print("\n--- With Normalization (your implementation) ---")
w_yes, b_yes, hist_yes = gradient_descent_vectorized(
    X_tr_norm2, y_tr_raw, learning_rate=0.01, epochs=1000)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: Without Normalization only
hist_no_clean = [c for c in hist_no if not np.isnan(c)]
axes[0].plot(hist_no_clean, color='steelblue')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cost J(w,b)')
axes[0].set_title('Without Normalization\n(diverges after epoch {})'.format(len(hist_no_clean)))
axes[0].grid(True, alpha=0.3)

# Middle: With Normalization only
axes[1].plot(hist_yes, color='tomato')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Cost J(w,b)')
axes[1].set_title('With Normalization\n(converges)')
axes[1].grid(True, alpha=0.3)

# Right: Test MSE comparison
mse_no  = np.mean((predict_vectorized(X_te_raw,   w_no,  b_no)  - y_te_raw) ** 2)
mse_yes = np.mean((predict_vectorized(X_te_norm2, w_yes, b_yes) - y_te_raw) ** 2)
axes[2].bar(['Without Norm', 'With Norm'], [mse_no, mse_yes],
            color=['steelblue', 'tomato'], alpha=0.8, edgecolor='black')
axes[2].set_ylabel('Test MSE')
axes[2].set_title('Test MSE Comparison (Multiple LR)')
axes[2].grid(True, alpha=0.3, axis='y')
for i, v in enumerate([mse_no, mse_yes]):
    axes[2].text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

print(f"Test MSE without normalization: {mse_no:.4f}")
print(f"Test MSE with    normalization: {mse_yes:.4f}")

## Part 4-3. Overfitting with Polynomial Features

Before applying regularization, let's first observe **overfitting**.  
By expanding a single feature (MedInc) into high-degree polynomial features,  
the model becomes too complex and fits the training data too well — but fails on test data.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Define MedInc single feature from already-scaled X_train_m
X_medinc_train = X_train_m[:, 0:1]
n_samples = 50
idx = np.random.choice(len(X_medinc_train), n_samples, replace=False)
X_medinc_small = X_medinc_train[idx]
y_small = y_train_m[idx]

degrees = [1, 5, 10]
train_mses, test_mses = [], []
poly_models = {}

X_medinc_test = X_test_m[:, 0:1]

for deg in degrees:
    poly = PolynomialFeatures(degree=deg, include_bias=False)
    X_poly_train = poly.fit_transform(X_medinc_small)
    X_poly_test  = poly.transform(X_medinc_test)

    scaler = StandardScaler()
    X_poly_train = scaler.fit_transform(X_poly_train)
    X_poly_test  = scaler.transform(X_poly_test)

    # Add bias column manually AFTER scaling
    X_poly_train_b = np.hstack([np.ones((X_poly_train.shape[0], 1)), X_poly_train])
    X_poly_test_b  = np.hstack([np.ones((X_poly_test.shape[0],  1)), X_poly_test])

    w_p, _, _, _ = np.linalg.lstsq(X_poly_train_b, y_small, rcond=None)

    train_mse = np.mean((X_poly_train_b @ w_p - y_small) ** 2)
    test_mse  = np.mean((X_poly_test_b  @ w_p - y_test_m) ** 2)
    train_mses.append(train_mse)
    test_mses.append(test_mse)

    poly_models[deg] = (poly, scaler, w_p)
    print(f"degree={deg:2d} | Train MSE: {train_mse:.4f} | Test MSE: {test_mse:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Train vs Test MSE by degree
axes[0].plot(degrees, train_mses, 'o-', label='Train MSE', color='steelblue')
axes[0].plot(degrees, test_mses,  'o-', label='Test MSE',  color='tomato')
axes[0].set_xlabel('Polynomial Degree')
axes[0].set_ylabel('MSE')
axes[0].set_title('Overfitting: Train vs Test MSE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: Fitted curves
X_plot = np.linspace(X_medinc_small.min(), X_medinc_small.max(), 200).reshape(-1, 1)
colors = ['steelblue', 'green', 'tomato']
for deg, col in zip(degrees, colors):
    poly, scaler, w_p = poly_models[deg]
    X_poly_plot = scaler.transform(poly.transform(X_plot))
    X_poly_plot_b = np.hstack([np.ones((X_poly_plot.shape[0], 1)), X_poly_plot])
    y_plot = X_poly_plot_b @ w_p
    axes[1].plot(X_plot, y_plot, label=f'degree={deg}', color=col)

axes[1].scatter(X_medinc_small, y_small,
                alpha=0.3, s=15, color='gray', label='data (sample)')
axes[1].set_xlabel('MedInc (standardized)')
axes[1].set_ylabel('Price ($100K)')
axes[1].set_title('Fitted Curves by Degree')
axes[1].set_ylim(-1, 8)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 4-4. Regularization

Regularization is a technique to prevent **overfitting** by penalizing large weight values.
We use L2 regularization (Ridge) that adds a penalty term to the cost function:

$$J(\vec{w}, b) = \frac{1}{2m}\left[\sum_{i=1}^{m}\left(f_{\vec{w},b}\left(\vec{x}^{(i)}\right) - y^{(i)}\right)^2 + \frac{\lambda}{2m}\sum_{j=1}^{n}w_j^2\right]$$

where the first term is the MSE loss and the second term is the L2 penalty.

This modifies the gradient update for **w** (but **not b**):

$$\text{repeat } \{$$

$$w_j = w_j - \alpha\left[\frac{1}{m}\sum_{i=1}^{m}\left(f_{\vec{w},b}\left(\vec{x}^{(i)}\right) - y^{(i)}\right)x_j^{(i)} + \frac{\lambda}{m}w_j\right]$$

$$b = b - \alpha\left[\frac{1}{m}\sum_{i=1}^{m}\left(f_{\vec{w},b}\left(\vec{x}^{(i)}\right) - y^{(i)}\right)\right]$$

$$\}\text{ simultaneous update for } j = 1 \ldots n$$

**λ (lambda)** controls regularization strength:
- λ = 0 → no regularization (same as before)
- Large λ → stronger penalty → weights shrink toward zero

> ⚠️ Regularization is applied to `w` only. **Never regularize the bias term `b`.**

In [ ]:
def compute_cost_regularized(y_hat, y, w, lambda_):
    """
    Compute L2 regularized MSE cost.
    Arguments:
        y_hat:   (m,) predicted values
        y:       (m,) ground truth values
        w:       (n,) weight vector
        lambda_: regularization strength (scalar)
    Returns:
        cost: scalar

    Formula:
        cost = (1/2m) * sum((y_hat - y)^2)  +  (lambda_ / 2m) * sum(w^2)
    """
    m = y.shape[0]
    mse_cost = 0.0
    reg_term = 0.0

    # Step 1: Compute MSE cost (same as compute_cost_vectorized)
    # YOUR CODE HERE

    # Step 2: Compute the regularization term (lambda_ / (2*m)) * np.sum(w**2)
    # YOUR CODE HERE

    return mse_cost + reg_term


def compute_gradients_regularized(X, y_hat, y, w, lambda_):
    """
    Compute gradients with L2 regularization.
    Arguments:
        X:       (m, n) feature matrix
        y_hat:   (m,) predicted values
        y:       (m,) ground truth values
        w:       (n,) weight vector
        lambda_: regularization strength (scalar)
    Returns:
        dj_dw: (n,) gradient w.r.t. weights  (with regularization)
        dj_db: scalar gradient w.r.t. bias   (NO regularization on b)

    Formula:
        dj_dw = (1/m) * X.T @ (y_hat - y)  +  (lambda_ / m) * w
        dj_db = (1/m) * sum(y_hat - y)
    """
    m = X.shape[0]
    dj_dw = np.zeros(X.shape[1])
    dj_db = 0.0

    # Step 1: Compute base gradients (same as compute_gradients_vectorized)
    # YOUR CODE HERE

    # Step 2: Add regularization term to dj_dw only (NOT to dj_db)
    # YOUR CODE HERE

    return dj_dw, dj_db


def gradient_descent_regularized(X, y, learning_rate=0.01, epochs=1000, lambda_=0.0):
    """
    Gradient descent with L2 regularization for multiple linear regression.
    (You do not need to modify this function.)
    """
    n = X.shape[1]
    w = np.zeros(n)
    b = 0.0
    cost_history = []
    mse_history = []

    for epoch in range(epochs):
        y_hat = predict_vectorized(X, w, b)
        cost  = compute_cost_regularized(y_hat, y, w, lambda_)
        mse   = compute_cost_vectorized(y_hat, y)
        cost_history.append(cost)
        mse_history.append(mse)
        dj_dw, dj_db = compute_gradients_regularized(X, y_hat, y, w, lambda_)
        w = w - learning_rate * dj_dw
        b = b - learning_rate * dj_db
        if epoch % 100 == 0:
            print(f"Epoch {epoch:4d}: Cost {cost:.6f}")

    return w, b, cost_history, mse_history

### Part 4-4a. Sanity Check (do not modify)

In [ ]:
# Sanity check — do not modify this cell
np.random.seed(0)
m_d, n_d = 5, 3
X_d = np.random.randn(m_d, n_d)
w_d = np.array([1.0, 2.0, -1.0])
b_d = 0.5
y_d = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_hat_d = predict_vectorized(X_d, w_d, b_d)
lambda_d = 1.0

# --- cost check ---
cost_reg = compute_cost_regularized(y_hat_d, y_d, w_d, lambda_d)
expected_mse = np.sum((y_hat_d - y_d)**2) / (2 * m_d)
expected_reg = (lambda_d / (2 * m_d)) * np.sum(w_d**2)
expected_cost = expected_mse + expected_reg
assert np.isclose(cost_reg, expected_cost), \
    f"compute_cost_regularized: got {cost_reg:.6f}, expected {expected_cost:.6f}"

# --- gradient check (w) ---
dj_dw_reg, dj_db_reg = compute_gradients_regularized(X_d, y_hat_d, y_d, w_d, lambda_d)
expected_dj_dw = X_d.T @ (y_hat_d - y_d) / m_d + (lambda_d / m_d) * w_d
expected_dj_db = np.sum(y_hat_d - y_d) / m_d
assert np.allclose(dj_dw_reg, expected_dj_dw), \
    f"compute_gradients_regularized: dj_dw wrong\n  got:      {dj_dw_reg}\n  expected: {expected_dj_dw}"
assert np.isclose(dj_db_reg, expected_dj_db), \
    f"compute_gradients_regularized: dj_db wrong (got {dj_db_reg:.6f}, expected {expected_dj_db:.6f})"

# --- lambda_=0 should equal unregularized ---
cost_noreg = compute_cost_regularized(y_hat_d, y_d, w_d, lambda_=0.0)
assert np.isclose(cost_noreg, expected_mse), \
    f"When lambda_=0, regularized cost should equal MSE cost"

print("✅ L2 Regularization: All checks passed!")

### Part 4-4b. Experiment: Effect of λ on Overfitting

In [ ]:
# Use degree=10 polynomial (overfitted model from Part 4-3a)
poly_10, scaler_10, _ = poly_models[10]  # ← _ 하나만

X_poly_train_10 = scaler_10.transform(poly_10.transform(X_medinc_small))  # ← small
X_poly_test_10  = scaler_10.transform(poly_10.transform(X_medinc_test))

# Add bias column (same as training)
X_poly_train_10 = np.hstack([np.ones((X_poly_train_10.shape[0], 1)), X_poly_train_10])
X_poly_test_10  = np.hstack([np.ones((X_poly_test_10.shape[0],  1)), X_poly_test_10])

lambdas = [0.0, 10.0, 100.0, 1000.0]
results_reg = {}

for lam in lambdas:
    print(f"\n--- λ = {lam} ---")
    w_r, b_r, hist_r, mse_hist_r = gradient_descent_regularized(
        X_poly_train_10, y_small, learning_rate=0.01, epochs=2000, lambda_=lam)  # ← y_small
    train_mse = np.mean((predict_vectorized(X_poly_train_10, w_r, b_r) - y_small) ** 2)
    test_mse  = np.mean((predict_vectorized(X_poly_test_10,  w_r, b_r) - y_test_m) ** 2)
    results_reg[lam] = {'w': w_r, 'b': b_r, 'history': hist_r,
                        'train_mse': train_mse, 'test_mse': test_mse, 'mse_history': mse_hist_r}
    print(f"  Train MSE: {train_mse:.4f} | Test MSE: {test_mse:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['steelblue', 'tomato', 'green', 'darkorange']

# Left: weight magnitude per lambda
for (lam, res), col in zip(results_reg.items(), colors):
    axes[0].bar(np.arange(len(res['w'])) + list(results_reg.keys()).index(lam) * 0.2,
                np.abs(res['w']), 0.2, label=f'λ={lam}', color=col, alpha=0.8)
axes[0].set_xlabel('Weight index')
axes[0].set_ylabel('|weight|')
axes[0].set_title('Weight Magnitude by λ\n(λ↑ → weights shrink toward 0)')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Middle: Train vs Test MSE per lambda
train_mses_reg = [res['train_mse'] for res in results_reg.values()]
test_mses_reg  = [res['test_mse']  for res in results_reg.values()]
x_pos = np.arange(len(lambdas))
width = 0.35
axes[1].bar(x_pos - width/2, train_mses_reg, width, label='Train MSE', color='steelblue', alpha=0.8)
axes[1].bar(x_pos + width/2, test_mses_reg,  width, label='Test MSE',  color='tomato',    alpha=0.8)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'λ={l}' for l in lambdas])
axes[1].set_ylabel('MSE')
axes[1].set_title('Train vs Test MSE by λ')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# Right: Fitted curves per lambda
X_plot = np.linspace(X_medinc_train.min(), X_medinc_train.max(), 200).reshape(-1, 1)
X_poly_plot = scaler_10.transform(poly_10.transform(X_plot))
X_poly_plot = np.hstack([np.ones((X_poly_plot.shape[0], 1)), X_poly_plot])

axes[2].scatter(X_medinc_small, y_small,
                alpha=0.3, s=15, color='gray', label='data (sample)')
for (lam, res), col in zip(results_reg.items(), colors):
    y_plot = predict_vectorized(X_poly_plot, res['w'], res['b'])
    axes[2].plot(X_plot, y_plot, label=f'λ={lam}', color=col, linewidth=1.5)
axes[2].set_xlabel('MedInc (standardized)')
axes[2].set_ylabel('Price ($100K)')
axes[2].set_title('Fitted Curves by λ\n(degree=10 polynomial)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim(-1, 8)

plt.tight_layout()
plt.show()

print("\nSummary:")
print(f"{'λ':>8} | {'Train MSE':>10} | {'Test MSE':>10}")
print("-" * 35)
for lam, res in results_reg.items():
    print(f"{lam:>8.1f} | {res['train_mse']:>10.4f} | {res['test_mse']:>10.4f}")

---
## Part 5. Analysis Questions

**Instructions:** Please provide detailed answers to the following questions based on your experimental results. Your analysis is as important as your code!

### Q1. Correlation Analysis
Which feature has the strongest positive correlation with housing price? Explain intuitively why this makes sense.

### Q2. Learning Rate Sensitivity
Re-run Simple LR with `learning_rate` set to `0.001`, `0.01`, and `0.5`.
Describe how the cost curve changes in each case, and explain how you would choose an appropriate learning rate.

### Q3. Simple vs Multiple LR
Compare the two models using the test error (MSE: Mean Squared Error) and the Predicted vs Actual plot.
Why does Multiple LR outperform Simple LR?
Also, identify the features with the largest and smallest learned weights — do they match the correlation results from Q1?

### Q4. Limitations of Linear Regression
The Predicted vs Actual plot shows that predictions are not perfect.
In what price range does the model perform poorly?
Based on what you observed in Part 4-3, what type of model might better capture the relationship between features and housing price?

### Q5. Effect of Feature Normalization
Run the code in Part 4-2b and compare the two runs (with and without normalization).
- Without normalization, what happens to the cost during training?
- With normalization, does the model converge successfully at the same learning rate?
- Looking at the feature statistics printed in Part 4-2b, which feature do you think caused the problem? Why?

### Q6. Effect of L2 Regularization
Based on the results from Part 4-4b, how do the learned weight values change as λ increases?
Which λ gives the best test MSE, and does regularization improve performance compared to λ=0?
What would happen to the weights and predictions if λ were set to an extremely large value (e.g., 10000)?